# Myocardial Infarction Prediction Using ECG (PTB-XL)
### Improved Version
Changes from previous version:
- Added **Undersampling** (60:40 non-MI:MI ratio) on train set only
- Added **10-Fold Cross Validation** using PTB-XL strat_fold
- Added **XGBoost** model
- Added **PR-AUC** metric
- Added **Threshold tuning** for recall-sensitive diagnosis
- Explicit **Hyperparameter settings** for all models
- Saved train/test IDs for reproducibility

In [3]:
import os
import ast
import json
import numpy as np
import pandas as pd
from tqdm import tqdm

import wfdb
import pywt                          # Wavelet features
from numpy.fft import fft            # FFT features

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import (
    roc_auc_score, f1_score, precision_score,
    recall_score, average_precision_score,
    precision_recall_curve
)
from xgboost import XGBClassifier

# ── Constants ──────────────────────────────────────────────────────────────────
DATA_DIR     = r"ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3"
USE_500HZ    = True           # True = 500 Hz (5000 timesteps) — lebih detail
RANDOM_STATE = 42
UNDERSAMPLE_RATIO = 1.0      # non-MI : MI  →  60:40

CACHE_DIR = os.path.join(DATA_DIR, "_cache_features")
os.makedirs(CACHE_DIR, exist_ok=True)

REPRO_DIR = "reproducibility"
os.makedirs(REPRO_DIR, exist_ok=True)

print("All imports OK")


All imports OK


## 1. Load Data

In [4]:
def load_ptbxl(data_dir: str):
    db_path  = os.path.join(data_dir, "ptbxl_database.csv")
    scp_path = os.path.join(data_dir, "scp_statements.csv")

    df  = pd.read_csv(db_path)
    scp = pd.read_csv(scp_path, index_col=0)

    df["scp_codes"] = df["scp_codes"].apply(ast.literal_eval)
    return df, scp


def build_mi_label(df: pd.DataFrame, scp: pd.DataFrame) -> pd.DataFrame:
    mi_codes = scp[scp["diagnostic_class"] == "MI"].index.tolist()

    def has_mi(scp_codes):
        return any(code in mi_codes for code in scp_codes.keys())

    df["is_mi"] = df["scp_codes"].apply(has_mi).astype(int)
    return df


df, scp = load_ptbxl(DATA_DIR)
df = build_mi_label(df, scp)

# ── Class distribution (original) ─────────────────────────────────────────────
total = len(df)
mi_count    = df["is_mi"].sum()
non_mi_count = total - mi_count
print(f"Total recordings : {total}")
print(f"MI               : {mi_count}  ({mi_count/total*100:.1f}%)")
print(f"Non-MI           : {non_mi_count}  ({non_mi_count/total*100:.1f}%)")

Total recordings : 21799
MI               : 5469  (25.1%)
Non-MI           : 16330  (74.9%)


## 2. Train / Test Split (Patient-Wise via strat_fold)

In [5]:
# PTB-XL strat_fold guarantees patient-wise separation:
# folds 1-9 = training, fold 10 = held-out test
# Patients do NOT appear in both train and test sets.

full_train_df = df[df["strat_fold"].between(1, 9)].copy()
test_df       = df[df["strat_fold"] == 10].copy()

print(f"Full train set : {len(full_train_df)} (MI={full_train_df['is_mi'].sum()})")
print(f"Test set       : {len(test_df)}  (MI={test_df['is_mi'].sum()})")
print("\nNOTE: Test set is kept with ORIGINAL distribution (no undersampling).")

Full train set : 19601 (MI=4919)
Test set       : 2198  (MI=550)

NOTE: Test set is kept with ORIGINAL distribution (no undersampling).


## 3. Undersampling (Train Set Only — 60:40 non-MI:MI)

In [6]:
def undersample_train(df: pd.DataFrame, ratio: float, random_state: int) -> pd.DataFrame:
    """
    Undersample the majority class (non-MI) in the training set.

    Parameters
    ----------
    df           : training DataFrame (already split from test)
    ratio        : desired non-MI / MI ratio  (e.g. 1.5 → 60:40)
    random_state : for reproducibility

    Returns
    -------
    Undersampled DataFrame (shuffled)
    """
    mi_df     = df[df["is_mi"] == 1]
    non_mi_df = df[df["is_mi"] == 0]

    n_keep = int(len(mi_df) * ratio)          # how many non-MI to keep
    n_keep = min(n_keep, len(non_mi_df))      # cannot exceed available

    non_mi_sampled = non_mi_df.sample(n=n_keep, random_state=random_state)
    balanced_df    = pd.concat([mi_df, non_mi_sampled]).sample(
        frac=1, random_state=random_state
    ).reset_index(drop=True)

    return balanced_df


train_df = undersample_train(full_train_df, UNDERSAMPLE_RATIO, RANDOM_STATE)

mi_t    = train_df["is_mi"].sum()
non_mi_t = len(train_df) - mi_t
print(f"Undersampled train : {len(train_df)} total")
print(f"  MI               : {mi_t}  ({mi_t/len(train_df)*100:.1f}%)")
print(f"  Non-MI           : {non_mi_t}  ({non_mi_t/len(train_df)*100:.1f}%)")

# ── Save IDs for reproducibility ──────────────────────────────────────────────
with open(os.path.join(REPRO_DIR, "train_ecg_ids.json"), "w") as f:
    json.dump(train_df["ecg_id"].tolist(), f)
with open(os.path.join(REPRO_DIR, "test_ecg_ids.json"), "w") as f:
    json.dump(test_df["ecg_id"].tolist(), f)

print("\nReproducibility IDs saved to ./reproducibility/")

Undersampled train : 9838 total
  MI               : 4919  (50.0%)
  Non-MI           : 4919  (50.0%)

Reproducibility IDs saved to ./reproducibility/


## 4. Feature Extraction (Statistical Features per Lead)

In [7]:
# ── Feature extraction: Statistical + FFT + Wavelet ──────────────────────────

def extract_statistical_features(signal: np.ndarray) -> list:
    """10 statistical features per lead."""
    return [
        np.mean(signal),
        np.std(signal),
        np.min(signal),
        np.max(signal),
        np.median(signal),
        np.percentile(signal, 25),
        np.percentile(signal, 75),
        np.var(signal),
        np.sqrt(np.mean(signal ** 2)),   # RMS
        np.max(signal) - np.min(signal)  # Range
    ]


def extract_fft_features(signal: np.ndarray) -> list:
    """
    4 FFT (frequency-domain) features per lead.
    Captures dominant frequency patterns in ECG — useful for
    detecting abnormal rhythm associated with MI.
    """
    fft_vals = np.abs(fft(signal))[:len(signal) // 2]
    return [
        np.mean(fft_vals),      # Average spectral energy
        np.std(fft_vals),       # Spectral variability
        np.max(fft_vals),       # Peak frequency magnitude
        float(np.argmax(fft_vals))  # Dominant frequency index
    ]


def extract_wavelet_features(signal: np.ndarray, wavelet: str = "db4", level: int = 4) -> list:
    """
    15 Wavelet features per lead (3 stats × 5 sub-bands).
    Wavelet decomposition captures both time AND frequency info,
    making it ideal for detecting QRS complex and ST changes in MI.
    Uses Daubechies-4 wavelet — standard for ECG analysis.
    """
    coeffs = pywt.wavedec(signal, wavelet=wavelet, level=level)
    feats = []
    for coeff in coeffs:  # 5 sub-bands: [cA4, cD4, cD3, cD2, cD1]
        feats += [
            np.mean(np.abs(coeff)),   # Mean absolute coefficient
            np.std(coeff),            # Std deviation
            np.sum(coeff ** 2)        # Energy
        ]
    return feats


def extract_lead_features(signals: np.ndarray) -> list:
    """
    Extract all features from 12-lead ECG signal.

    Per lead:
      - 10 statistical features
      -  4 FFT features
      - 15 Wavelet features (db4, level=4)
      = 29 features × 12 leads = 348 total features

    Parameters
    ----------
    signals : np.ndarray of shape (timesteps, 12)
    """
    all_feats = []
    for lead_idx in range(signals.shape[1]):
        s = signals[:, lead_idx]
        all_feats.extend(extract_statistical_features(s))
        all_feats.extend(extract_fft_features(s))
        all_feats.extend(extract_wavelet_features(s))
    return all_feats


def build_feature_matrix(
    df: pd.DataFrame,
    data_dir: str,
    use_500hz: bool,
    cache_name: str,
    tabular_features: list
):
    hz_tag     = "500hz" if use_500hz else "100hz"
    tab_tag    = "_".join(tabular_features) if tabular_features else "no_tabular"
    cache_file = os.path.join(CACHE_DIR, f"{cache_name}_{hz_tag}_{tab_tag}_v2.npz")

    if os.path.exists(cache_file):
        print(f"Loading cached features: {cache_file}")
        data = np.load(cache_file)
        return data["X"], data["y"]

    print(f"Extracting features for {len(df)} samples...")
    features_list, labels = [], []

    for _, row in tqdm(df.iterrows(), total=len(df)):
        filename = row["filename_hr"] if use_500hz else row["filename_lr"]
        filepath = os.path.normpath(os.path.join(data_dir, filename))

        if not os.path.exists(filepath + ".hea"):
            continue
        try:
            signals, _ = wfdb.rdsamp(filepath)
            features   = extract_lead_features(signals)   # 348 ECG features

            for feat in tabular_features:
                features.append(row[feat] if feat in row.index else 0)

            features_list.append(features)
            labels.append(row["is_mi"])
        except Exception as e:
            print(f"Error: {filepath} → {e}")

    X = np.array(features_list)
    y = np.array(labels, dtype=int)

    np.savez(cache_file, X=X, y=y)
    print(f"Saved cache: {cache_file}")
    return X, y


print(f"Feature extraction functions defined.")
print(f"Features per sample: 29 × 12 leads = 348 ECG features")


Feature extraction functions defined.
Features per sample: 29 × 12 leads = 348 ECG features


## 5. Model Definitions (with Explicit Hyperparameters)

In [8]:
# ── Hyperparameter settings (explicitly defined for reproducibility) ────────────

LR_PARAMS = dict(
    C            = 1.0,          # Regularisation strength (inverse)
    solver       = "liblinear",  # Optimiser
    class_weight = "balanced",   # Handles remaining imbalance
    max_iter     = 5000,
    random_state = RANDOM_STATE
)

SVM_PARAMS = dict(
    kernel       = "rbf",        # Radial Basis Function kernel
    C            = 1.0,          # Regularisation
    gamma        = "scale",      # Kernel coefficient
    class_weight = "balanced",
    probability  = True,         # Needed for predict_proba
    random_state = RANDOM_STATE
)

RF_PARAMS = dict(
    n_estimators     = 200,      # Number of trees
    max_depth        = None,     # Grow until pure leaves
    min_samples_split= 2,
    min_samples_leaf = 1,
    max_features     = "sqrt",   # Features considered per split
    class_weight     = "balanced",
    random_state     = RANDOM_STATE,
    n_jobs           = -1
)

XGB_PARAMS = dict(
    n_estimators     = 200,
    max_depth        = 6,
    learning_rate    = 0.1,
    subsample        = 0.8,
    colsample_bytree = 0.8,
    scale_pos_weight = 1,        # Will be overridden per fold based on ratio
    eval_metric      = "logloss",
    random_state     = RANDOM_STATE,
    n_jobs           = -1
)


def make_models(scale_pos_weight: float = 1.0):
    """Return fresh model instances. scale_pos_weight is for XGBoost."""
    xgb_p = {**XGB_PARAMS, "scale_pos_weight": scale_pos_weight}
    return {
        "Logistic Regression": Pipeline([
            ("scaler", StandardScaler()),
            ("clf",    LogisticRegression(**LR_PARAMS))
        ]),
        "SVM": Pipeline([
            ("scaler", StandardScaler()),
            ("clf",    SVC(**SVM_PARAMS))
        ]),
        "Random Forest": Pipeline([
            ("scaler", StandardScaler()),
            ("clf",    RandomForestClassifier(**RF_PARAMS))
        ]),
        "XGBoost": Pipeline([
            ("scaler", StandardScaler()),
            ("clf",    XGBClassifier(**xgb_p))
        ])
    }

print("Model hyperparameters defined.")

Model hyperparameters defined.


## 6. Evaluation Functions (F1, AUC, PR-AUC, Threshold Tuning)

In [9]:
def find_threshold_for_min_recall(y_true, y_prob, min_recall: float = 0.85):
    """
    Find the HIGHEST threshold that still achieves at least `min_recall`.
    This is clinically meaningful: we want to catch as many MI cases as
    possible (high recall) while maximising precision.

    Parameters
    ----------
    min_recall : minimum acceptable recall (default 0.85)

    Returns
    -------
    best_threshold, achieved_recall, achieved_precision
    """
    precisions, recalls, thresholds = precision_recall_curve(y_true, y_prob)

    best_threshold = 0.5
    best_precision = 0.0
    best_recall    = 0.0

    for prec, rec, thr in zip(precisions[:-1], recalls[:-1], thresholds):
        if rec >= min_recall and prec > best_precision:
            best_precision = prec
            best_recall    = rec
            best_threshold = thr

    return best_threshold, best_recall, best_precision


def evaluate_probs(y_true, y_prob, threshold: float = 0.5):
    """
    Evaluate model predictions at a given threshold.
    Includes: Precision, Recall, F1, ROC-AUC, PR-AUC.
    """
    y_pred = (y_prob >= threshold).astype(int)

    return {
        "Threshold" : threshold,
        "Precision" : precision_score(y_true, y_pred, zero_division=0),
        "Recall"    : recall_score(y_true, y_pred, zero_division=0),
        "F1"        : f1_score(y_true, y_pred, zero_division=0),
        "ROC-AUC"   : roc_auc_score(y_true, y_prob),
        "PR-AUC"    : average_precision_score(y_true, y_prob),   # NEW
    }


def evaluate_with_tuned_threshold(y_true, y_prob, min_recall: float = 0.85):
    """
    Evaluate at both default (0.5) and recall-tuned threshold.
    Returns dict with both result sets.
    """
    default_results = evaluate_probs(y_true, y_prob, threshold=0.5)

    tuned_thr, _, _ = find_threshold_for_min_recall(y_true, y_prob, min_recall)
    tuned_results   = evaluate_probs(y_true, y_prob, threshold=tuned_thr)
    tuned_results["Threshold"] = round(tuned_thr, 3)

    return default_results, tuned_results

print("Evaluation functions defined.")

Evaluation functions defined.


## 7. 10-Fold Cross Validation (using strat_fold)

In [10]:
def run_cross_validation(
    full_train_df : pd.DataFrame,
    tabular_features: list,
    n_folds: int = 9
):
    """
    10-Fold CV using PTB-XL strat_fold (folds 1-9 rotated).
    Each fold: train on 8 folds, validate on 1 fold.
    Undersampling is applied INSIDE each train fold to prevent leakage.
    """
    cv_results = {name: [] for name in ["Logistic Regression", "SVM", "Random Forest", "XGBoost"]}

    for val_fold in range(1, n_folds + 1):
        print(f"\n── Fold {val_fold}/{n_folds} ──")

        fold_train_df = full_train_df[full_train_df["strat_fold"] != val_fold].copy()
        fold_val_df   = full_train_df[full_train_df["strat_fold"] == val_fold].copy()

        # Undersample only the train portion
        fold_train_df = undersample_train(fold_train_df, UNDERSAMPLE_RATIO, RANDOM_STATE)

        X_tr, y_tr = build_feature_matrix(
            fold_train_df, DATA_DIR, USE_500HZ,
            cache_name=f"cv_train_fold{val_fold}",
            tabular_features=tabular_features
        )
        X_val, y_val = build_feature_matrix(
            fold_val_df, DATA_DIR, USE_500HZ,
            cache_name=f"cv_val_fold{val_fold}",
            tabular_features=tabular_features
        )

        # scale_pos_weight for XGBoost based on current fold ratio
        neg = (y_tr == 0).sum()
        pos = (y_tr == 1).sum()
        spw = neg / pos if pos > 0 else 1.0

        models = make_models(scale_pos_weight=spw)

        for model_name, model in models.items():
            model.fit(X_tr, y_tr)
            y_prob = model.predict_proba(X_val)[:, 1]
            metrics = evaluate_probs(y_val, y_prob, threshold=0.5)
            metrics["fold"] = val_fold
            cv_results[model_name].append(metrics)
            print(f"  {model_name:20s} → F1={metrics['F1']:.4f}  PR-AUC={metrics['PR-AUC']:.4f}")

    return cv_results


def summarise_cv(cv_results: dict):
    """Print mean ± std for each metric across folds."""
    metric_cols = ["Precision", "Recall", "F1", "ROC-AUC", "PR-AUC"]
    rows = []
    for model_name, fold_metrics in cv_results.items():
        fold_df = pd.DataFrame(fold_metrics)
        row = {"Model": model_name}
        for col in metric_cols:
            row[col] = f"{fold_df[col].mean():.4f} ± {fold_df[col].std():.4f}"
        rows.append(row)
    return pd.DataFrame(rows).set_index("Model")

print("Cross-validation functions defined.")

Cross-validation functions defined.


## 7b. XGBoost Hyperparameter Tuning (RandomizedSearchCV)


In [11]:
# ── XGBoost Hyperparameter Tuning ────────────────────────────────────────────
# Run this ONCE to find best params, then plug into XGB_PARAMS above.
# Uses 3-fold CV on training set (fast version: n_iter=20)

from sklearn.model_selection import RandomizedSearchCV

def tune_xgboost(X_train, y_train, n_iter=20, cv=3):
    """
    RandomizedSearchCV for XGBoost hyperparameter tuning.
    Optimises for F1 score (recall-precision balance).

    Parameters
    ----------
    n_iter : number of random combinations to try (more = better but slower)
    cv     : number of cross-validation folds for tuning
    """
    param_dist = {
        "clf__n_estimators"      : [100, 200, 300, 500],
        "clf__max_depth"          : [3, 4, 5, 6, 8],
        "clf__learning_rate"      : [0.01, 0.05, 0.1, 0.2],
        "clf__subsample"          : [0.6, 0.7, 0.8, 1.0],
        "clf__colsample_bytree"   : [0.6, 0.7, 0.8, 1.0],
        "clf__min_child_weight"   : [1, 3, 5],
        "clf__gamma"              : [0, 0.1, 0.2, 0.5],
    }

    spw = (y_train == 0).sum() / max((y_train == 1).sum(), 1)
    base_pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", XGBClassifier(
            scale_pos_weight = spw,
            eval_metric      = "logloss",
            random_state     = RANDOM_STATE,
            n_jobs           = -1
        ))
    ])

    search = RandomizedSearchCV(
        base_pipeline,
        param_distributions = param_dist,
        n_iter              = n_iter,
        scoring             = "f1",
        cv                  = cv,
        random_state        = RANDOM_STATE,
        n_jobs              = -1,
        verbose             = 1
    )
    search.fit(X_train, y_train)

    print(f"\nBest F1 (CV): {search.best_score_:.4f}")
    print(f"Best params : {search.best_params_}")
    return search.best_params_


# ── Build feature matrix dulu untuk tuning ────────────────────────────────────
print("Building feature matrix for XGBoost tuning...")
X_tune, y_tune = build_feature_matrix(
    train_df, DATA_DIR, USE_500HZ,
    cache_name="final_train", tabular_features=[]  # ← ganti TABULAR_FEATURES dengan []
)

print("\nRunning XGBoost hyperparameter tuning (n_iter=20)...")
print("This may take 5-15 minutes depending on your machine.\n")
best_xgb_params = tune_xgboost(X_tune, y_tune, n_iter=20, cv=3)

# Update XGB_PARAMS global dengan hasil tuning
for key, val in best_xgb_params.items():
    clean_key = key.replace("clf__", "")
    XGB_PARAMS[clean_key] = val

print("\nXGB_PARAMS updated with best values:")
for k, v in XGB_PARAMS.items():
    print(f"  {k}: {v}")


Building feature matrix for XGBoost tuning...
Loading cached features: ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3\_cache_features\final_train_500hz_no_tabular_v2.npz

Running XGBoost hyperparameter tuning (n_iter=20)...
This may take 5-15 minutes depending on your machine.

Fitting 3 folds for each of 20 candidates, totalling 60 fits


d:\Anaconda\envs\deep_learning\Lib\site-packages\sklearn\model_selection\_validation.py:490: FitFailedWarning: 
1 fits failed out of a total of 60.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
1 fits failed with the following error:
Traceback (most recent call last):
  File "d:\Anaconda\envs\deep_learning\Lib\site-packages\sklearn\model_selection\_validation.py", line 833, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
    ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\Anaconda\envs\deep_learning\Lib\site-packages\sklearn\base.py", line 1336, in wrapper
    return fit_method(estimator, *args, **kwargs)
  File "d:\Anaconda\envs\deep_learning\Lib\site-packages\sklearn\pipeline.py", line 621, in fit
    sel


Best F1 (CV): 0.7254
Best params : {'clf__subsample': 0.7, 'clf__n_estimators': 300, 'clf__min_child_weight': 3, 'clf__max_depth': 3, 'clf__learning_rate': 0.1, 'clf__gamma': 0.2, 'clf__colsample_bytree': 0.7}

XGB_PARAMS updated with best values:
  n_estimators: 300
  max_depth: 3
  learning_rate: 0.1
  subsample: 0.7
  colsample_bytree: 0.7
  scale_pos_weight: 1
  eval_metric: logloss
  random_state: 42
  n_jobs: -1
  min_child_weight: 3
  gamma: 0.2


## 8. Run Cross Validation

In [12]:
TABULAR_FEATURES = ["age"]

print("Starting 9-Fold Cross Validation (folds 1-9)...")
print("This may take a while on first run (feature extraction + caching).\n")

cv_results = run_cross_validation(full_train_df, TABULAR_FEATURES, n_folds=9)

print("\n" + "="*60)
print("CROSS VALIDATION SUMMARY (mean ± std across 9 folds)")
print("="*60)
cv_summary = summarise_cv(cv_results)
print(cv_summary.to_string())

Starting 9-Fold Cross Validation (folds 1-9)...
This may take a while on first run (feature extraction + caching).


── Fold 1/9 ──
Loading cached features: ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3\_cache_features\cv_train_fold1_500hz_age_v2.npz
Loading cached features: ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3\_cache_features\cv_val_fold1_500hz_age_v2.npz
  Logistic Regression  → F1=0.6036  PR-AUC=0.5835
  SVM                  → F1=0.6135  PR-AUC=0.6544
  Random Forest        → F1=0.5926  PR-AUC=0.6329
  XGBoost              → F1=0.6119  PR-AUC=0.6833

── Fold 2/9 ──
Loading cached features: ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3\_cache_features\cv_train_fold2_500hz_age_v2.npz
Loading cached features: ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3\_cache_features\cv_val_fold2_500hz_age_v2.npz
  Logistic Regression  → F1=0.5932  PR-AUC=0.5639
  SVM                  → F1=0.6129  PR-AUC

100%|██████████| 2192/2192 [01:57<00:00, 18.59it/s]


Saved cache: ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3\_cache_features\cv_val_fold3_500hz_age_v2.npz
  Logistic Regression  → F1=0.6285  PR-AUC=0.6368
  SVM                  → F1=0.6147  PR-AUC=0.6737
  Random Forest        → F1=0.6361  PR-AUC=0.6634
  XGBoost              → F1=0.6519  PR-AUC=0.7280

── Fold 4/9 ──
Extracting features for 8736 samples...


100%|██████████| 8736/8736 [06:21<00:00, 22.88it/s]


Saved cache: ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3\_cache_features\cv_train_fold4_500hz_age_v2.npz
Extracting features for 2174 samples...


...

Saved cache: ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3\_cache_features\cv_val_fold8_500hz_age_v2.npz
  Logistic Regression  → F1=0.5938  PR-AUC=0.5815
  SVM                  → F1=0.6100  PR-AUC=0.6602
  Random Forest        → F1=0.6054  PR-AUC=0.6315
  XGBoost              → F1=0.6362  PR-AUC=0.7014

── Fold 9/9 ──
Extracting features for 8758 samples...


100%|██████████| 8758/8758 [04:07<00:00, 35.39it/s]


Saved cache: ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3\_cache_features\cv_train_fold9_500hz_age_v2.npz
Extracting features for 2183 samples...


100%|██████████| 2183/2183 [01:02<00:00, 34.86it/s]


Saved cache: ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3\_cache_features\cv_val_fold9_500hz_age_v2.npz
  Logistic Regression  → F1=0.6085  PR-AUC=0.5673
  SVM                  → F1=0.6257  PR-AUC=0.6493
  Random Forest        → F1=0.6000  PR-AUC=0.6129
  XGBoost              → F1=0.6404  PR-AUC=0.6787

CROSS VALIDATION SUMMARY (mean ± std across 9 folds)
                           Precision           Recall               F1          ROC-AUC           PR-AUC
Model                                                                                                   
Logistic Regression  0.5021 ± 0.0153  0.7645 ± 0.0229  0.6059 ± 0.0131  0.8243 ± 0.0113  0.5828 ± 0.0235
SVM                  0.5287 ± 0.0238  0.7440 ± 0.0467  0.6167 ± 0.0088  0.8429 ± 0.0100  0.6511 ± 0.0155
Random Forest        0.5228 ± 0.0421  0.7333 ± 0.0811  0.6060 ± 0.0176  0.8385 ± 0.0124  0.6360 ± 0.0151
XGBoost              0.5362 ± 0.0122  0.7850 ± 0.0232  0.6370 ± 0.0128  0.8612 ± 0.0107  0.693

## 9. Final Evaluation on Held-Out Test Set

In [13]:
print("Building final train/test feature matrices...")

X_train, y_train = build_feature_matrix(
    train_df, DATA_DIR, USE_500HZ,
    cache_name="final_train", tabular_features=TABULAR_FEATURES
)
X_test, y_test = build_feature_matrix(
    test_df, DATA_DIR, USE_500HZ,
    cache_name="final_test",  tabular_features=TABULAR_FEATURES
)

print(f"X_train: {X_train.shape}  |  y_train MI rate: {y_train.mean():.3f}")
print(f"X_test : {X_test.shape}   |  y_test  MI rate: {y_test.mean():.3f}")
print("\nNOTE: Test MI rate reflects original (imbalanced) distribution.")

Building final train/test feature matrices...
Extracting features for 9838 samples...


100%|██████████| 9838/9838 [04:05<00:00, 40.09it/s]


Saved cache: ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3\_cache_features\final_train_500hz_age_v2.npz
Extracting features for 2198 samples...


100%|██████████| 2198/2198 [01:34<00:00, 23.38it/s]

Saved cache: ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3\_cache_features\final_test_500hz_age_v2.npz
X_train: (9838, 349)  |  y_train MI rate: 0.500
X_test : (2198, 349)   |  y_test  MI rate: 0.250

NOTE: Test MI rate reflects original (imbalanced) distribution.


In [14]:
# ── Train final models & evaluate ─────────────────────────────────────────────
MIN_RECALL_TARGET = 0.85   # Clinical target: catch at least 85% of MI cases

spw = (y_train == 0).sum() / max((y_train == 1).sum(), 1)
final_models = make_models(scale_pos_weight=spw)

results_default = {}
results_tuned   = {}

for model_name, model in final_models.items():
    print(f"Training {model_name}...")
    model.fit(X_train, y_train)
    y_prob = model.predict_proba(X_test)[:, 1]

    default_res, tuned_res = evaluate_with_tuned_threshold(
        y_test, y_prob, min_recall=MIN_RECALL_TARGET
    )
    results_default[model_name] = default_res
    results_tuned[model_name]   = tuned_res

print("\nDone.")

Training Logistic Regression...
Training SVM...
Training Random Forest...
Training XGBoost...

Done.


## 10. Results Summary

In [15]:
metric_cols = ["Threshold", "Precision", "Recall", "F1", "ROC-AUC", "PR-AUC"]

df_default = pd.DataFrame(results_default).T[metric_cols]
df_tuned   = pd.DataFrame(results_tuned).T[metric_cols]

print("=" * 70)
print("FINAL TEST RESULTS — Default Threshold (0.5)")
print("=" * 70)
print(df_default.round(4).to_string())

print()
print("=" * 70)
print(f"FINAL TEST RESULTS — Recall-Tuned Threshold (min recall ≥ {MIN_RECALL_TARGET})")
print("=" * 70)
print(df_tuned.round(4).to_string())

print()
print("NOTE: Tuned threshold prioritises recall (sensitivity) for clinical safety.")
print("A lower threshold means the model flags more cases as MI, reducing missed diagnoses.")

FINAL TEST RESULTS — Default Threshold (0.5)
                     Threshold  Precision  Recall      F1  ROC-AUC  PR-AUC
Logistic Regression        0.5     0.4880  0.7018  0.5757   0.8002  0.5454
SVM                        0.5     0.5158  0.7109  0.5979   0.8236  0.6188
Random Forest              0.5     0.4940  0.7527  0.5965   0.8254  0.6178
XGBoost                    0.5     0.5300  0.7218  0.6112   0.8430  0.6603

FINAL TEST RESULTS — Recall-Tuned Threshold (min recall ≥ 0.85)
                     Threshold  Precision  Recall      F1  ROC-AUC  PR-AUC
Logistic Regression      0.325     0.3988  0.8564  0.5442   0.8002  0.5454
SVM                      0.346     0.4369  0.8564  0.5786   0.8236  0.6188
Random Forest            0.400     0.4116  0.8545  0.5556   0.8254  0.6178
XGBoost                  0.356     0.4519  0.8545  0.5912   0.8430  0.6603

NOTE: Tuned threshold prioritises recall (sensitivity) for clinical safety.
A lower threshold means the model flags more cases as MI, reduc

## 11. Hyperparameter Summary Table (for Paper)

In [16]:
hyperparam_summary = pd.DataFrame([
    {
        "Model"            : "Logistic Regression",
        "Key Hyperparameters": f"C={LR_PARAMS['C']}, solver={LR_PARAMS['solver']}, "
                               f"class_weight={LR_PARAMS['class_weight']}, max_iter={LR_PARAMS['max_iter']}"
    },
    {
        "Model"            : "SVM",
        "Key Hyperparameters": f"kernel={SVM_PARAMS['kernel']}, C={SVM_PARAMS['C']}, "
                               f"gamma={SVM_PARAMS['gamma']}, class_weight={SVM_PARAMS['class_weight']}"
    },
    {
        "Model"            : "Random Forest",
        "Key Hyperparameters": f"n_estimators={RF_PARAMS['n_estimators']}, max_depth={RF_PARAMS['max_depth']}, "
                               f"max_features={RF_PARAMS['max_features']}, class_weight={RF_PARAMS['class_weight']}"
    },
    {
        "Model"            : "XGBoost",
        "Key Hyperparameters": f"n_estimators={XGB_PARAMS['n_estimators']}, max_depth={XGB_PARAMS['max_depth']}, "
                               f"learning_rate={XGB_PARAMS['learning_rate']}, subsample={XGB_PARAMS['subsample']}"
    },
]).set_index("Model")

print("HYPERPARAMETER SETTINGS")
print(hyperparam_summary.to_string())

HYPERPARAMETER SETTINGS
                                                                            Key Hyperparameters
Model                                                                                          
Logistic Regression               C=1.0, solver=liblinear, class_weight=balanced, max_iter=5000
SVM                                       kernel=rbf, C=1.0, gamma=scale, class_weight=balanced
Random Forest        n_estimators=200, max_depth=None, max_features=sqrt, class_weight=balanced
XGBoost                         n_estimators=300, max_depth=3, learning_rate=0.1, subsample=0.7
